# Modul 07: Optimierung und Regression mit NumPy

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Optimierung verstehen, Regression mit NumPy  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene Grundlagen  
    **Orientierungszeit:** etwa 110 bis 150 Minuten

    ## Überblick

    Sie untersuchen Verlustfunktionen zunächst für einzelne Parameter und implementieren anschließend eine lineare Regression vollständig mit NumPy. Raster, Gradientenabstieg, Lernrate, Abbruch, Validierung, geschlossene Lösung und Residuen werden verglichen.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_07A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_07B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Eine Verlustfunktion für einen einzelnen Modellparameter beschreiben.
- Raster- und Gradientenabstiegssuche für einfache Funktionen implementieren.
- Lernrate, Abbruchbedingungen, Skalierung und Verlustkurven analysieren.
- Das lineare Modell y = w*x + b und seine Vorhersagen implementieren.
- Gewicht und Bias mit Gradientenabstieg auf kleinen Daten trainieren.
- Baseline, trainiertes Modell, geschlossene Lösung und Polynomvariante vergleichen.

    ## Bewertete Fähigkeiten

    - Verlustfunktion, endliche Differenz und analytischer Gradient
- Gradientenabstieg, Lernratenvergleich und Early Stopping
- lineare Regression aus NumPy-Grundoperationen
- Validierungsverlust, Normalgleichung, Residuen und Polynommerkmale

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

x_all = np.linspace(-3.0, 5.0, 90)
y_all = 2.7 * x_all - 1.4 + rng.normal(0, 1.0, size=x_all.size)

# Ein zweiter Datensatz enthält eine gekrümmte Beziehung.
x_curve = np.linspace(-3.0, 3.0, 100)
y_curve = 1.5 + 0.8 * x_curve - 1.2 * x_curve**2 + rng.normal(0, 0.8, x_curve.size)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Verlust über ein Werteraster untersuchen

    Betrachten Sie das vereinfachte Modell `y_hat = w * x` ohne Bias für die Daten `x_small` und `y_small`.

1. Schreiben Sie eine Funktion für den mittleren quadratischen Fehler.
2. Berechnen Sie den Verlust für 121 Werte von `w` zwischen -1 und 5.
3. Bestimmen Sie den besten Rasterwert.
4. Schätzen Sie die lokale Steigung am besten Rasterwert mit einer zentralen endlichen Differenz.
5. Zeichnen Sie Verlust gegen `w` und markieren Sie das Minimum.

> **Hinweis:** Das Minimum des Rasters hängt von Bereich und Auflösung des Rasters ab.

In [ ]:
x_small = np.array([1.0, 2.0, 3.0, 4.0])
y_small = np.array([2.2, 4.1, 6.4, 8.0])

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Verlust über ein Werteraster untersuchen
#
# Ziel dieser Codezelle:
# Betrachten Sie das vereinfachte Modell yhat = w x ohne Bias für die Daten xsmall
# und ysmall. 1. Schreiben Sie eine Funktion für den mittleren quadratischen Fehler.
# 2. Berechnen Sie den Verlust für 121 Werte von w zwis...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

x_small = np.array([1.0, 2.0, 3.0, 4.0])
y_small = np.array([2.2, 4.1, 6.4, 8.0])

def mse_for_weight(weight: float) -> float:
    # Das Modell berechnet für jede Eingabe eine Vorhersage.
    predictions = weight * x_small

    # Der quadratische Fehler wird über alle Beobachtungen gemittelt.
    return float(np.mean((y_small - predictions) ** 2))

weight_grid = np.linspace(-1.0, 5.0, 121)
grid_losses = np.array([mse_for_weight(w) for w in weight_grid])

best_index = int(np.argmin(grid_losses))
best_grid_weight = float(weight_grid[best_index])
best_grid_loss = float(grid_losses[best_index])

# Eine zentrale Differenz vergleicht den Verlust leicht rechts und links
# des aktuellen Punktes. Ein kleines epsilon approximiert die Ableitung.
epsilon = 1e-5
numerical_slope = (
    mse_for_weight(best_grid_weight + epsilon)
    - mse_for_weight(best_grid_weight - epsilon)
) / (2 * epsilon)

print("Bester Rasterwert:", best_grid_weight)
print("Rasterverlust:", round(best_grid_loss, 6))
print("Numerische Steigung am Rasterminimum:", round(numerical_slope, 6))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(weight_grid, grid_losses)
ax.scatter([best_grid_weight], [best_grid_loss], s=80, label="Rasterminimum")
ax.set_title("MSE als Funktion des Gewichts")
ax.set_xlabel("Gewicht w")
ax.set_ylabel("MSE")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 1

Das Raster findet nur das beste der geprüften Gewichte. Eine feinere Schrittweite erhöht die Genauigkeit, benötigt aber mehr Auswertungen. Die lokale Steigung liegt am diskreten Rasterminimum ungefähr, aber nicht zwingend exakt bei null. Gradientenverfahren nutzen die Steigung direkt und können kontinuierliche Werte erreichen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Gradientenabstieg und Lernrate vergleichen

    Minimieren Sie die Funktion `loss(theta) = (theta - 3.5)**2 + 1` mit Gradientenabstieg.

1. Implementieren Sie Gradient und Trainingsschleife.
2. Stoppen Sie, wenn der Betrag des Gradienten kleiner als `1e-6` ist oder maximal 100 Schritte erreicht sind.
3. Vergleichen Sie Lernraten 0.05, 0.25 und 1.10.
4. Zeichnen Sie jede Verlustkurve in einer eigenen Abbildung.
5. Erklären Sie Konvergenz, Oszillation oder Divergenz.

> **Hinweis:** Beobachten Sie sowohl den Verlust als auch den Parameterverlauf.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Gradientenabstieg und Lernrate vergleichen
#
# Ziel dieser Codezelle:
# Minimieren Sie die Funktion loss(theta) = (theta - 3.5)2 + 1 mit
# Gradientenabstieg. 1. Implementieren Sie Gradient und Trainingsschleife. 2.
# Stoppen Sie, wenn der Betrag des Gradienten kleiner als 1e-6 ist oder maxima...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

def quadratic_loss(theta: float) -> float:
    return (theta - 3.5) ** 2 + 1.0

def quadratic_gradient(theta: float) -> float:
    # Ableitung von (theta - 3.5)^2 ist 2*(theta - 3.5).
    return 2.0 * (theta - 3.5)

def run_gradient_descent(
    learning_rate: float,
    initial_theta: float = -5.0,
    max_steps: int = 100,
    gradient_tolerance: float = 1e-6,
) -> pd.DataFrame:
    theta = float(initial_theta)
    history = []

    for step in range(max_steps):
        gradient = quadratic_gradient(theta)
        current_loss = quadratic_loss(theta)
        history.append(
            {
                "step": step,
                "theta": theta,
                "loss": current_loss,
                "gradient": gradient,
            }
        )

        # Der Stopp wird vor einem weiteren Update geprüft.
        if abs(gradient) < gradient_tolerance:
            break

        theta = theta - learning_rate * gradient

    return pd.DataFrame(history)

learning_rates = [0.05, 0.25, 1.10]
histories = {
    rate: run_gradient_descent(rate)
    for rate in learning_rates
}

for rate, history in histories.items():
    last = history.iloc[-1]
    print(
        f"Lernrate {rate:.2f}: Schritte={len(history)}, "
        f"theta={last['theta']:.5f}, loss={last['loss']:.5f}"
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history["step"], history["loss"], marker="o", markersize=3)
    ax.set_title(f"Verlustkurve bei Lernrate {rate:.2f}")
    ax.set_xlabel("Schritt")
    ax.set_ylabel("Verlust")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Reflexion zu Aufgabe 2

Eine kleine Lernrate nähert sich dem Minimum langsam. Die Lernrate 0,25 konvergiert bei dieser einfachen Funktion schnell. Eine Lernrate über 1 kann den Parameter auf die jeweils andere Seite des Minimums springen lassen und die Abstände vergrößern. Der geeignete Bereich hängt von der Krümmung und Skalierung der Verlustfunktion ab.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Lineare Regression mit Gewicht und Bias trainieren

    Teilen Sie `x_all` und `y_all` chronologisch nach ihrem vorhandenen Index in 70 Prozent Training und 30 Prozent Validierung. Implementieren Sie:

- `predict_linear(x, w, b)`,
- MSE,
- analytische Gradienten für `w` und `b`,
- eine Trainingsschleife mit 1.000 Epochen und Lernrate 0.03.

Speichern Sie Trainings- und Validierungsverlust alle 10 Epochen und geben Sie die gelernten Parameter aus.

> **Hinweis:** Vergessen Sie beim Bias-Gradienten nicht den Mittelwert über alle Fehler.

In [ ]:
split_index = int(0.70 * len(x_all))
x_train, x_validation = x_all[:split_index], x_all[split_index:]
y_train, y_validation = y_all[:split_index], y_all[split_index:]

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Lineare Regression mit Gewicht und Bias trainieren
#
# Ziel dieser Codezelle:
# Teilen Sie xall und yall chronologisch nach ihrem vorhandenen Index in 70 Prozent
# Training und 30 Prozent Validierung. Implementieren Sie: - predictlinear(x, w, b),
# - MSE, - analytische Gradienten für w und b, - eine...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

split_index = int(0.70 * len(x_all))
x_train, x_validation = x_all[:split_index], x_all[split_index:]
y_train, y_validation = y_all[:split_index], y_all[split_index:]

# Die Eingaben werden mit Trainingsstatistiken skaliert. Das verbessert
# die Optimierung und verhindert, dass Validierungsdaten den Fit beeinflussen.
x_mean = x_train.mean()
x_std = x_train.std(ddof=0)
x_train_scaled = (x_train - x_mean) / x_std
x_validation_scaled = (x_validation - x_mean) / x_std

def predict_linear(x: np.ndarray, weight: float, bias: float) -> np.ndarray:
    return weight * x + bias

def mean_squared_error_numpy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    return float(np.mean((y_true - y_pred) ** 2))

def linear_gradients(
    x: np.ndarray,
    y_true: np.ndarray,
    weight: float,
    bias: float,
) -> tuple[float, float]:
    predictions = predict_linear(x, weight, bias)
    errors = predictions - y_true

    # Der Faktor 2 folgt aus der Ableitung des Quadrats.
    gradient_weight = float(2.0 * np.mean(errors * x))
    gradient_bias = float(2.0 * np.mean(errors))
    return gradient_weight, gradient_bias

weight = 0.0
bias = 0.0
learning_rate = 0.03
epochs = 1000
history_rows = []

for epoch in range(epochs):
    gradient_w, gradient_b = linear_gradients(
        x_train_scaled,
        y_train,
        weight,
        bias,
    )

    weight -= learning_rate * gradient_w
    bias -= learning_rate * gradient_b

    if epoch % 10 == 0 or epoch == epochs - 1:
        train_predictions = predict_linear(x_train_scaled, weight, bias)
        validation_predictions = predict_linear(
            x_validation_scaled,
            weight,
            bias,
        )
        history_rows.append(
            {
                "epoch": epoch,
                "train_mse": mean_squared_error_numpy(
                    y_train,
                    train_predictions,
                ),
                "validation_mse": mean_squared_error_numpy(
                    y_validation,
                    validation_predictions,
                ),
            }
        )

training_history = pd.DataFrame(history_rows)

# Parameter werden zurück in die Originalskala übersetzt.
original_weight = weight / x_std
original_bias = bias - weight * x_mean / x_std

print("Gewicht in Originalskala:", round(original_weight, 3))
print("Bias in Originalskala:", round(original_bias, 3))
print(training_history.tail())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(training_history["epoch"], training_history["train_mse"], label="Training")
ax.plot(training_history["epoch"], training_history["validation_mse"], label="Validierung")
ax.set_title("Training und Validierung")
ax.set_xlabel("Epoche")
ax.set_ylabel("MSE")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 3

Die Skalierung macht die Lernrate leichter kontrollierbar. Die gelernten Parameter in Originaleinheiten sollten ungefähr bei der erzeugenden Geraden liegen, unterscheiden sich aber wegen des Rauschens. Da der Split nach Index erfolgt und `x_all` sortiert ist, liegt die Validierung in einem anderen Wertebereich. Das prüft Extrapolation und kann einen höheren Validierungsfehler erzeugen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Baseline, geschlossene Lösung und Residuen vergleichen

    Vergleichen Sie auf den Validierungsdaten:

1. eine Mittelwert-Baseline aus `y_train`,
2. Ihr Gradientenabstiegsmodell,
3. die geschlossene lineare Lösung mit einer Designmatrix aus Einsen und `x_train`.

Berechnen Sie MAE und RMSE. Zeichnen Sie die Daten, beide Regressionsgeraden und anschließend ein eigenes Residuenplot für das beste lineare Modell.

> **Hinweis:** Vergleichen Sie Modelle auf genau denselben Validierungsbeobachtungen.

In [ ]:
# Nutzen Sie Variablen aus Aufgabe 3. Falls nötig, führen Sie Aufgabe 3 zuerst aus.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Baseline, geschlossene Lösung und Residuen vergleichen
#
# Ziel dieser Codezelle:
# Vergleichen Sie auf den Validierungsdaten: 1. eine Mittelwert-Baseline aus ytrain,
# 2. Ihr Gradientenabstiegsmodell, 3. die geschlossene lineare Lösung mit einer
# Designmatrix aus Einsen und xtrain. Berechnen Sie MAE un...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Die Baseline sagt für jede Validierungsbeobachtung denselben
# Trainingsmittelwert voraus.
baseline_predictions = np.full_like(y_validation, y_train.mean())

# Das Gradientenmodell wird mit den in Aufgabe 3 gelernten Parametern
# direkt in Originaleinheiten ausgewertet.
gradient_predictions = predict_linear(
    x_validation,
    original_weight,
    original_bias,
)

# Die Designmatrix enthält eine Einsenspalte für den Bias und eine
# Spalte für x. pinv ist robust gegenüber singulären Matrizen.
train_design = np.column_stack([np.ones_like(x_train), x_train])
closed_parameters = np.linalg.pinv(train_design) @ y_train
closed_bias, closed_weight = closed_parameters

validation_design = np.column_stack(
    [np.ones_like(x_validation), x_validation]
)
closed_predictions = validation_design @ closed_parameters

def mae_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float]:
    residuals = y_true - y_pred
    return (
        float(np.mean(np.abs(residuals))),
        float(np.sqrt(np.mean(residuals**2))),
    )

methods = {
    "Mittelwert-Baseline": baseline_predictions,
    "Gradientenabstieg": gradient_predictions,
    "Geschlossene Lösung": closed_predictions,
}
rows = []
for method_name, predictions in methods.items():
    mae, rmse = mae_rmse(y_validation, predictions)
    rows.append({"method": method_name, "MAE": mae, "RMSE": rmse})

comparison = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
best_method = comparison.loc[0, "method"]
best_predictions = methods[best_method]

print(comparison.round(3).to_string(index=False))
print("Geschlossene Parameter:", round(closed_weight, 3), round(closed_bias, 3))

x_line = np.linspace(x_all.min(), x_all.max(), 200)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_train, y_train, alpha=0.65, label="Training")
ax.scatter(x_validation, y_validation, alpha=0.85, label="Validierung")
ax.plot(x_line, original_weight * x_line + original_bias, label="Gradientenabstieg")
ax.plot(x_line, closed_weight * x_line + closed_bias, linestyle="--", label="Geschlossen")
ax.set_title("Lineare Regressionsmodelle")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

residuals = y_validation - best_predictions
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(best_predictions, residuals, s=65)
ax.axhline(0, linewidth=1)
ax.set_title(f"Validierungsresiduen: {best_method}")
ax.set_xlabel("Vorhersage")
ax.set_ylabel("Residuum")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 4

Gradientenabstieg und geschlossene Lösung sollten bei korrekter Implementierung ähnliche Geraden ergeben. Kleine Unterschiede entstehen durch endliche Optimierungsschritte und numerische Details. Ein systematisches Muster in den Residuen deutet darauf hin, dass die lineare Form wichtige Struktur nicht erfasst. Die Baseline zeigt, wie viel Nutzen die Eingabevariable gegenüber einer konstanten Vorhersage bringt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Polynomiale Erweiterung

    Modellieren Sie den gekrümmten Datensatz `x_curve`, `y_curve` mit:

- einer linearen Designmatrix `[1, x]`,
- einer quadratischen Designmatrix `[1, x, x**2]`.

Verwenden Sie für beide die geschlossene Lösung. Teilen Sie die ersten 70 Prozent als Training und die letzten 30 Prozent als Test. Vergleichen Sie MAE und RMSE und visualisieren Sie Daten und beide Kurven. Erklären Sie, warum die quadratische Variante hier angemessen ist und warum immer höhere Grade nicht automatisch besser sind.

> **Hinweis:** Polynomiale Regression bleibt linear in ihren Parametern, obwohl sie nichtlinear in x ist.

In [ ]:
curve_split = int(0.70 * len(x_curve))
x_curve_train, x_curve_test = x_curve[:curve_split], x_curve[curve_split:]
y_curve_train, y_curve_test = y_curve[:curve_split], y_curve[curve_split:]

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Polynomiale Erweiterung
#
# Ziel dieser Codezelle:
# Modellieren Sie den gekrümmten Datensatz xcurve, ycurve mit: - einer linearen
# Designmatrix [1, x], - einer quadratischen Designmatrix [1, x, x2]. Verwenden Sie
# für beide die geschlossene Lösung. Teilen Sie die ersten...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

curve_split = int(0.70 * len(x_curve))
x_curve_train, x_curve_test = x_curve[:curve_split], x_curve[curve_split:]
y_curve_train, y_curve_test = y_curve[:curve_split], y_curve[curve_split:]

def design_matrix(x: np.ndarray, degree: int) -> np.ndarray:
    # column_stack erzeugt [1, x, x^2, ...] bis zum gewünschten Grad.
    return np.column_stack([x**power for power in range(degree + 1)])

prediction_sets = {}
comparison_rows = []

for degree, name in [(1, "linear"), (2, "quadratisch")]:
    X_train_design = design_matrix(x_curve_train, degree)
    X_test_design = design_matrix(x_curve_test, degree)

    parameters = np.linalg.pinv(X_train_design) @ y_curve_train
    predictions = X_test_design @ parameters
    prediction_sets[name] = (degree, parameters, predictions)

    residuals = y_curve_test - predictions
    comparison_rows.append(
        {
            "model": name,
            "MAE": float(np.mean(np.abs(residuals))),
            "RMSE": float(np.sqrt(np.mean(residuals**2))),
        }
    )

polynomial_comparison = pd.DataFrame(comparison_rows).sort_values("RMSE")
print(polynomial_comparison.round(3).to_string(index=False))

x_plot = np.linspace(x_curve.min(), x_curve.max(), 300)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_curve_train, y_curve_train, alpha=0.55, label="Training")
ax.scatter(x_curve_test, y_curve_test, alpha=0.8, label="Test")

for name, (degree, parameters, _) in prediction_sets.items():
    y_plot = design_matrix(x_plot, degree) @ parameters
    ax.plot(x_plot, y_plot, label=name)

ax.set_title("Lineare und quadratische Regression")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 5

Die Daten wurden mit einem quadratischen Term erzeugt, daher kann `[1, x, x²]` die grundlegende Form abbilden. Das lineare Modell zeigt systematische Abweichungen. Höhere Polynomgrade erhöhen jedoch Flexibilität und können Rauschen statt Struktur lernen, besonders bei wenigen Daten oder außerhalb des Trainingsbereichs stark schwanken. Modellkomplexität muss deshalb mit unabhängigen Daten bewertet werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.